# NB14 ? Estrategias Avanzadas: IPS+TD, Multi-Behavior, LightGCN, Ensemble Spearman

## Auditor?a
| Notebook | Modelo | NDCG@10 |
|---|---|---|
| NB12b | RP3beta+TD ? GANADOR | **0.02859** |
| Target | ? | **0.03000** |

### Estrategias
1. IPS + Temporal Decay
2. Multi-Behavior + TD (Optuna)
3. LightGCN + TD
4. Ensemble inteligente por correlaci?n Spearman


In [ ]:
import sys, time, math, pickle, gc, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path
from itertools import combinations
from scipy.stats import spearmanr
from sklearn.preprocessing import normalize as skl_normalize
import torch
import torch.nn as nn
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

HERE     = Path().resolve()
ROOT     = HERE.parent if (HERE.parent / 'data').exists() else HERE
DATA_DIR = ROOT / 'data' / 'processed'
RAW_DIR  = ROOT / 'data' / 'raw'
ENC_DIR  = ROOT / 'encoders'
DOCS_DIR = ROOT / 'docs'

CUTOFF_DATE  = pd.Timestamp('2015-08-22', tz='UTC')
EASE_TOP     = 20_000
EASE_LAM_500 = 500.0
TD_DECAY     = 0.01
RP3_ALPHA    = 0.75
RP3_BETA     = 0.30
RANDOM_STATE = 42
K_VALUES     = [10]
NDCG_NB12B   = 0.02859
TARGET_NDCG  = 0.030

print(f'ROOT: {ROOT}')
print(f'PyTorch: {torch.__version__}  CUDA: {torch.cuda.is_available()}')
print(f'Ganador actual: NDCG@10={NDCG_NB12B}  Target: {TARGET_NDCG}')


In [ ]:
print('Cargando interaction_matrix.csv...')
t0 = time.time()
im = pd.read_csv(DATA_DIR / 'interaction_matrix.csv')
im['last_interaction_ts'] = pd.to_datetime(im['last_interaction_ts'], format='ISO8601', utc=True)
print(f'  IM: {im.shape}  [{time.time()-t0:.1f}s]')

train_mask = im['last_interaction_ts'] < CUTOFF_DATE
train_df   = im[train_mask].copy()
test_df    = im[~train_mask].copy()

warm_users  = sorted(set(train_df['visitorid'].unique()) & set(test_df['visitorid'].unique()))
rng         = np.random.default_rng(RANDOM_STATE)
N_EVAL      = 3000
eval_users  = rng.choice(warm_users, size=min(N_EVAL, len(warm_users)), replace=False).tolist()

test_items_by_user  = test_df.groupby('visitorid')['itemid'].apply(set).to_dict()
train_items_by_user = train_df.groupby('visitorid')['itemid'].apply(set).to_dict()
test_tx_by_user     = (
    test_df[test_df['last_interaction_type'] == 'transaction']
    .groupby('visitorid')['itemid'].apply(set).to_dict()
)

all_items_global = sorted(im['itemid'].unique())
n_items_global   = len(all_items_global)
n_test_tx        = len(test_df[test_df['last_interaction_type'] == 'transaction'])
baseline_conv    = n_test_tx / (len(set(test_df['visitorid'])) * n_items_global)

activity_groups = np.array([
    0 if len(train_items_by_user.get(u, set())) == 1
    else (1 if len(train_items_by_user.get(u, set())) <= 4 else 2)
    for u in eval_users
])
rng_split = np.random.default_rng(RANDOM_STATE)
val_mask  = np.zeros(len(eval_users), dtype=bool)
for g in [0, 1, 2]:
    idx_g = np.where(activity_groups == g)[0]
    if len(idx_g) == 0: continue
    n_val  = max(1, int(len(idx_g) * 0.15))
    chosen = rng_split.choice(idx_g, size=n_val, replace=False)
    val_mask[chosen] = True

eval_arr     = np.array(eval_users)
val_users    = eval_arr[val_mask].tolist()
test_users_b = eval_arr[~val_mask].tolist()

print(f'Train: {len(train_df):,}  Test: {len(test_df):,}')
print(f'val_users: {len(val_users):,}  test_users_b: {len(test_users_b):,}')


## 2 ? Matrices y funciones base

In [ ]:
all_train_users = sorted(train_df['visitorid'].unique())
all_train_items = sorted(train_df['itemid'].unique())
user2idx = {u: i for i, u in enumerate(all_train_users)}
item2idx = {it: i for i, it in enumerate(all_train_items)}
idx2item = {i: it for it, i in item2idx.items()}
n_u = len(all_train_users); n_i = len(all_train_items)

rows_r = train_df['visitorid'].map(user2idx).values
cols_r = train_df['itemid'].map(item2idx).values
vals_r = train_df['interaction_strength'].values.astype(np.float32)
R = sp.csr_matrix((vals_r, (rows_r, cols_r)), shape=(n_u, n_i), dtype=np.float32)

item_pop      = np.asarray(R.sum(axis=0)).ravel()
item_pop_dict = {idx2item[ix]: float(item_pop[ix]) for ix in range(n_i)}
n_total_train = float(R.sum())

top_items_idx    = np.argpartition(item_pop, -EASE_TOP)[-EASE_TOP:]
top_items_idx    = top_items_idx[np.argsort(item_pop[top_items_idx])[::-1]]
N_TOP            = len(top_items_idx)
top_items_global = [idx2item[ix] for ix in top_items_idx]
X_top_csr        = R[:, top_items_idx].astype(np.float32).tocsr()
pop_sub          = item_pop[top_items_idx].astype(np.float32)

print(f'R: {R.shape}  X_top: {X_top_csr.shape}')

def ndcg_at_k(r, rel, k):
    r = r[:k]
    dcg  = sum(1/math.log2(ii+2) for ii,x in enumerate(r) if x in rel)
    idcg = sum(1/math.log2(ii+2) for ii in range(min(len(rel), k)))
    return dcg/idcg if idcg else 0.

def prec_at_k(r, rel, k):  return len(set(r[:k]) & rel) / k if k else 0.
def rec_at_k(r, rel, k):   return len(set(r[:k]) & rel) / len(rel) if rel else 0.
def ap_at_k(r, rel, k):
    if not rel: return 0.
    s, h = 0., 0
    for ii, x in enumerate(r[:k]):
        if x in rel: h += 1; s += h/(ii+1)
    return s / min(len(rel), k)

def evaluate(get_fn, evals, tst, tst_tx, pop_d, nt, cat_sz, bconv, ks=K_VALUES):
    acc = {k: {m: [] for m in 'prnm'} for k in ks}
    seen = set(); ne = 0
    for uid in evals:
        ti = tst.get(uid, set())
        if not ti: continue
        try: recs = get_fn(uid, max(ks))
        except Exception: continue
        seen.update(recs); ne += 1
        for k in ks:
            acc[k]['p'].append(prec_at_k(recs, ti, k))
            acc[k]['r'].append(rec_at_k(recs,  ti, k))
            acc[k]['n'].append(ndcg_at_k(recs,  ti, k))
            acc[k]['m'].append(ap_at_k(recs,    ti, k))
    out = {'n_eval': ne}
    for k in ks:
        if not acc[k]['p']: continue
        out[f'NDCG@{k}']      = float(np.mean(acc[k]['n']))
        out[f'Precision@{k}'] = float(np.mean(acc[k]['p']))
        out[f'Recall@{k}']    = float(np.mean(acc[k]['r']))
        out[f'MAP@{k}']       = float(np.mean(acc[k]['m']))
    out['Coverage'] = len(seen) / cat_sz
    return out

def build_rp3(alpha, beta, X_csr, pop_arr):
    P_ui = skl_normalize(X_csr.astype(np.float64), norm='l1', axis=1)
    P_it = skl_normalize(X_csr.T.tocsr().astype(np.float64), norm='l1', axis=1)
    pop_beta = np.power(pop_arr + 1e-10, beta)
    W_raw = P_it @ P_ui
    W = np.asarray(W_raw.todense() if hasattr(W_raw, 'todense') else W_raw, dtype=np.float32)
    del W_raw
    np.power(W, alpha, out=W)
    W /= pop_beta[:, np.newaxis]
    np.fill_diagonal(W, 0.)
    return W

def make_get_rp3(W, X_csr, top_items_list):
    def get_fn(uid, n):
        if uid not in user2idx: return []
        ui  = user2idx[uid]
        row = X_csr.getrow(ui)
        x_u = np.asarray(row.todense(), dtype=np.float32).ravel()
        sc  = x_u @ W
        sc[x_u > 0] = -1.
        top = np.argpartition(sc, -n)[-n:]
        top = top[np.argsort(sc[top])[::-1]]
        return [top_items_list[ix] for ix in top]
    return get_fn

def minmax_norm(v):
    vmin, vmax = v.min(), v.max()
    r = vmax - vmin
    return (v - vmin) / r if r > 1e-12 else np.zeros_like(v)

print('Funciones base definidas: evaluate, build_rp3, make_get_rp3, minmax_norm')


## 3 ? Matriz base con Temporal Decay (decay=0.01, ?ptimo NB12b)

In [ ]:
print(f'Construyendo R_td con decay={TD_DECAY}...')
t0 = time.time()

train_df_td = train_df.copy()
train_df_td['days_before'] = (
    (CUTOFF_DATE - train_df_td['last_interaction_ts']).dt.total_seconds() / 86400.0
).clip(lower=0.0)

decay_w = np.exp(-TD_DECAY * train_df_td['days_before'].values)
vals_td = (train_df_td['interaction_strength'].values.astype(np.float32)
           * decay_w.astype(np.float32))

rows_td = train_df_td['visitorid'].map(user2idx).values
cols_td = train_df_td['itemid'].map(item2idx).values
R_td    = sp.csr_matrix((vals_td, (rows_td, cols_td)),
                         shape=(n_u, n_i), dtype=np.float32)

X_top_td = R_td[:, top_items_idx].astype(np.float32).tocsr()
pop_td   = np.asarray(R_td.sum(axis=0)).ravel()[top_items_idx].astype(np.float32)
print(f'  R_td: {R_td.shape}  [{time.time()-t0:.1f}s]')

print('Construyendo RP3beta+TD (ganador NB12b)...')
t0 = time.time()
W_rp3_td   = build_rp3(RP3_ALPHA, RP3_BETA, X_top_td, pop_td)
get_rp3_td = make_get_rp3(W_rp3_td, X_top_td, top_items_global)
print(f'  W_rp3_td: {W_rp3_td.shape}  [{time.time()-t0:.1f}s]')

t0 = time.time()
m_base_val = evaluate(
    get_rp3_td, val_users, test_items_by_user, test_tx_by_user,
    item_pop_dict, n_total_train, n_items_global, baseline_conv
)
ndcg_td_val = m_base_val['NDCG@10']
print(f'  RP3+TD val NDCG@10={ndcg_td_val:.5f}  [{time.time()-t0:.1f}s]')


## Estrategia 1 ? IPS + Temporal Decay

IPS (Inverse Propensity Score) debiasa la popularidad. Combinado con TD ataca dos sesgos distintos:
- TD: sesgo temporal (interacciones antiguas pesan menos)
- IPS: sesgo de popularidad (?tems poco vistos reciben m?s peso relativo)

```
propensity(i) = (pop(i) / max_pop)^smoothing
ips_weight(i) = 1 / propensity(i)
```


In [ ]:
def compute_ips_weights(matrix, smoothing=0.5):
    item_popularity = np.asarray(matrix.sum(axis=0)).flatten()
    propensity = (item_popularity / (item_popularity.max() + 1e-8)) ** smoothing
    ips_weights = 1.0 / (propensity + 1e-8)
    return ips_weights.astype(np.float32)

def apply_ips(matrix, ips_weights):
    return matrix.multiply(ips_weights)

smoothing_grid = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]

print('=' * 60)
print('ESTRATEGIA 1 ? IPS + Temporal Decay')
print(f'  Grid: {smoothing_grid}   val_users={len(val_users)}')
print('=' * 60)
print(f'{"smoothing":>10}  {"NDCG@10 val":>12}  {"Delta%":>10}  {"t(s)":>6}')

results_ips = []
for smoothing in smoothing_grid:
    t0 = time.time()
    ips_w     = compute_ips_weights(X_top_td, smoothing)
    X_td_ips  = apply_ips(X_top_td, ips_w).tocsr()
    pop_ips   = np.asarray(X_td_ips.sum(axis=0)).ravel().astype(np.float32)
    W_ips     = build_rp3(RP3_ALPHA, RP3_BETA, X_td_ips, pop_ips)
    get_ips   = make_get_rp3(W_ips, X_td_ips, top_items_global)
    m         = evaluate(
        get_ips, val_users, test_items_by_user, test_tx_by_user,
        item_pop_dict, n_total_train, n_items_global, baseline_conv
    )
    elapsed   = time.time() - t0
    ndcg_val  = m.get('NDCG@10', 0.)
    delta     = (ndcg_val - ndcg_td_val) / ndcg_td_val * 100 if ndcg_td_val > 0 else 0.
    results_ips.append({'smoothing': smoothing, 'NDCG@10_val': ndcg_val,
                        'delta': delta, 'W': W_ips, 'X': X_td_ips, 'ips_w': ips_w})
    del X_td_ips, pop_ips; gc.collect()
    print(f'  {smoothing:10.1f}  {ndcg_val:12.5f}  {"+" if delta>=0 else ""}{delta:9.1f}%  {elapsed:5.1f}s')

best_ips         = max(results_ips, key=lambda x: x['NDCG@10_val'])
best_smoothing   = best_ips['smoothing']
ndcg_ips_val     = best_ips['NDCG@10_val']
W_rp3_td_ips     = best_ips['W']
X_top_td_ips     = best_ips['X']
print(f'Mejor smoothing: {best_smoothing}  NDCG@10_val={ndcg_ips_val:.5f}')


In [ ]:
print(f'Evaluando RP3+TD+IPS(smoothing={best_smoothing}) en test...')
t0 = time.time()
get_ips_best = make_get_rp3(W_rp3_td_ips, X_top_td_ips, top_items_global)
m_ips_test   = evaluate(
    get_ips_best, test_users_b, test_items_by_user, test_tx_by_user,
    item_pop_dict, n_total_train, n_items_global, baseline_conv
)
print(f'  [{time.time()-t0:.1f}s]')
ndcg_ips_test = m_ips_test['NDCG@10']
d1 = (ndcg_ips_test - NDCG_NB12B) / NDCG_NB12B * 100
print(f'RP3+TD         : NDCG@10={NDCG_NB12B:.5f}')
print(f'RP3+TD+IPS     : NDCG@10={ndcg_ips_test:.5f}  ({"+" if d1>=0 else ""}{d1:.1f}%)')

import os
art_ips = {'model_name': f'RP3beta+TD+IPS(s={best_smoothing})',
           'alpha': RP3_ALPHA, 'beta': RP3_BETA, 'decay_rate': TD_DECAY,
           'smoothing': best_smoothing,
           'ndcg10_val': ndcg_ips_val, 'ndcg10_test': ndcg_ips_test,
           'W_rp3': W_rp3_td_ips,
           'top_items_global': top_items_global, 'top_items_idx': top_items_idx,
           'user2idx': user2idx}
out_ips = ENC_DIR / 'rp3beta_td_ips.pkl'
with open(out_ips, 'wb') as f:
    pickle.dump(art_ips, f, protocol=4)
print(f'Guardado: {out_ips}  ({os.path.getsize(out_ips)/1e6:.1f} MB)')


## Estrategia 2 ? Multi-Behavior + Temporal Decay

Aprender w_view, w_cart, w_trans con Optuna (40 trials) sobre datos con temporal decay.
Carga events.csv para acceder al tipo de evento por fila individual.


In [ ]:
print('Cargando events.csv...')
t0 = time.time()
events_raw = pd.read_csv(RAW_DIR / 'events.csv')
events_raw.columns = events_raw.columns.str.strip()
events_raw['ts'] = pd.to_datetime(events_raw['timestamp'] / 1000, unit='s', utc=True)
print(f'  events_raw: {events_raw.shape}  [{time.time()-t0:.1f}s]')
print(f'  Eventos: {events_raw["event"].value_counts().to_dict()}')

events_train = events_raw[events_raw['ts'] < CUTOFF_DATE].copy()
events_train['visitorid'] = events_train['visitorid'].astype(int)
events_train['itemid']    = events_train['itemid'].astype(int)
events_train = events_train[
    events_train['visitorid'].isin(user2idx) &
    events_train['itemid'].isin(item2idx)
].copy()
events_train['days_before'] = (
    (CUTOFF_DATE - events_train['ts']).dt.total_seconds() / 86400.0
).clip(lower=0.0)
events_train['decay_factor'] = np.exp(-TD_DECAY * events_train['days_before'].values)
print(f'  events_train filtrado: {len(events_train):,}')

def build_mb_matrix(events_df, w_view, w_cart, w_trans):
    behavior_map = {'view': w_view, 'addtocart': w_cart, 'transaction': w_trans}
    peso_tipo  = events_df['event'].map(behavior_map).fillna(w_view).values
    peso_final = peso_tipo * events_df['decay_factor'].values
    rows = events_df['visitorid'].map(user2idx).values
    cols = events_df['itemid'].map(item2idx).values
    R_mb = sp.csr_matrix((peso_final.astype(np.float32), (rows, cols)),
                          shape=(n_u, n_i), dtype=np.float32)
    X_mb  = R_mb[:, top_items_idx].tocsr().astype(np.float32)
    pop_mb = np.asarray(R_mb.sum(axis=0)).ravel()[top_items_idx].astype(np.float32)
    return X_mb, pop_mb

print('build_mb_matrix definido.')


In [ ]:
N_TRIALS_MB = 40

def objective_mb(trial):
    w_view  = trial.suggest_float('w_view',  0.1, 3.0)
    w_cart  = trial.suggest_float('w_cart',  1.0, 6.0)
    w_trans = trial.suggest_float('w_trans', 3.0, 12.0)
    X_mb, pop_mb = build_mb_matrix(events_train, w_view, w_cart, w_trans)
    W_mb  = build_rp3(RP3_ALPHA, RP3_BETA, X_mb, pop_mb)
    get_mb = make_get_rp3(W_mb, X_mb, top_items_global)
    m = evaluate(get_mb, val_users, test_items_by_user, test_tx_by_user,
                 item_pop_dict, n_total_train, n_items_global, baseline_conv)
    return m.get('NDCG@10', 0.)

print(f'Optuna Multi-Behavior ({N_TRIALS_MB} trials)...')
t_opt = time.time()
study_mb = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_mb.optimize(objective_mb, n_trials=N_TRIALS_MB, show_progress_bar=True)
print(f'Optuna MB finalizado en {time.time()-t_opt:.0f}s')

best_mb          = study_mb.best_params
best_mb_ndcg_val = study_mb.best_value
print(f'Pesos optimos: w_view={best_mb["w_view"]:.3f} w_cart={best_mb["w_cart"]:.3f} w_trans={best_mb["w_trans"]:.3f}')
print(f'NDCG@10_val={best_mb_ndcg_val:.5f}  Delta vs base={((best_mb_ndcg_val-ndcg_td_val)/ndcg_td_val*100):+.1f}%')

if best_mb['w_view'] < best_mb['w_cart'] < best_mb['w_trans']:
    print('Jerarquia w_view < w_cart < w_trans: intuitiva y esperada')
else:
    print('Jerarquia no estrictamente creciente ? posible ruido en RetailRocket')

trials_mb = sorted([t for t in study_mb.trials if t.value], key=lambda t: t.value, reverse=True)
print('Top 5 trials:')
for t in trials_mb[:5]:
    print(f'  NDCG@10_val={t.value:.5f}  {t.params}')


In [ ]:
print('Construyendo MB+TD con pesos optimos...')
t0 = time.time()
X_mb_best, pop_mb_best = build_mb_matrix(
    events_train, best_mb['w_view'], best_mb['w_cart'], best_mb['w_trans']
)
W_rp3_mb   = build_rp3(RP3_ALPHA, RP3_BETA, X_mb_best, pop_mb_best)
get_mb_best = make_get_rp3(W_rp3_mb, X_mb_best, top_items_global)
print(f'  [{time.time()-t0:.1f}s]')

print(f'Evaluando MB+TD en test ({len(test_users_b)})...')
t0 = time.time()
m_mb_test  = evaluate(
    get_mb_best, test_users_b, test_items_by_user, test_tx_by_user,
    item_pop_dict, n_total_train, n_items_global, baseline_conv
)
print(f'  [{time.time()-t0:.1f}s]')

ndcg_mb_test = m_mb_test['NDCG@10']
d2 = (ndcg_mb_test - NDCG_NB12B) / NDCG_NB12B * 100
print(f'RP3+TD    : NDCG@10={NDCG_NB12B:.5f}')
print(f'RP3+MB+TD : NDCG@10={ndcg_mb_test:.5f}  ({"+" if d2>=0 else ""}{d2:.1f}%)')

art_mb = {'model_name': 'RP3beta+MultiBehavior+TD',
          'alpha': RP3_ALPHA, 'beta': RP3_BETA, 'decay_rate': TD_DECAY,
          'w_view': best_mb['w_view'], 'w_cart': best_mb['w_cart'],
          'w_trans': best_mb['w_trans'],
          'ndcg10_val': best_mb_ndcg_val, 'ndcg10_test': ndcg_mb_test,
          'W_rp3': W_rp3_mb, 'optuna_best_params': best_mb,
          'top_items_global': top_items_global, 'top_items_idx': top_items_idx,
          'user2idx': user2idx}
out_mb = ENC_DIR / 'rp3beta_mb_td.pkl'
with open(out_mb, 'wb') as f:
    pickle.dump(art_mb, f, protocol=4)
import os; print(f'Guardado: {out_mb}  ({os.path.getsize(out_mb)/1e6:.1f} MB)')


## Estrategia 3 ? LightGCN + Temporal Decay

LightGCN (He et al. SIGIR 2020) aprende embeddings sobre el grafo bipartito ponderado por TD.

**Configuracion:** n_layers=1, embedding_dim=32, BPR loss, early stopping patience=5.

**Se?ales de alerta:**
- NDCG@10_val < 0.005 en epoch 30 ? documentar y pasar a Estrategia 4
- Tiempo/epoch > 20 min ? reducir batch_size


In [ ]:
class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, embedding_dim=32, n_layers=1):
        super().__init__()
        self.n_users   = n_users
        self.n_items   = n_items
        self.emb_dim   = embedding_dim
        self.n_layers  = n_layers
        self.user_emb  = nn.Embedding(n_users, embedding_dim)
        self.item_emb  = nn.Embedding(n_items, embedding_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def forward(self, adj_norm):
        E = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        all_emb = [E]
        for _ in range(self.n_layers):
            E = torch.sparse.mm(adj_norm, E)
            all_emb.append(E)
        return torch.stack(all_emb, dim=1).mean(dim=1).split([self.n_users, self.n_items])

    def bpr_loss(self, adj_norm, u_ids, pos_ids, neg_ids, l2_reg=1e-4):
        u_emb, i_emb = self.forward(adj_norm)
        u   = u_emb[u_ids];   pos = i_emb[pos_ids];  neg = i_emb[neg_ids]
        bpr = -torch.log(torch.sigmoid((u*pos).sum(1) - (u*neg).sum(1)) + 1e-8).mean()
        l2  = (self.user_emb.weight[u_ids].norm(2).pow(2)
               + self.item_emb.weight[pos_ids].norm(2).pow(2)
               + self.item_emb.weight[neg_ids].norm(2).pow(2)) / len(u_ids)
        return bpr + l2_reg * l2


def build_normalized_adj(R_matrix, n_u, n_i):
    R_coo    = R_matrix.tocoo()
    rows_ui  = R_coo.row;                  cols_ui  = R_coo.col + n_u
    rows_iu  = R_coo.col + n_u;            cols_iu  = R_coo.row
    data     = R_coo.data.astype(np.float32)
    all_rows = np.concatenate([rows_ui, rows_iu])
    all_cols = np.concatenate([cols_ui, cols_iu])
    all_data = np.concatenate([data, data])
    N        = n_u + n_i
    A        = sp.coo_matrix((all_data, (all_rows, all_cols)), shape=(N, N)).tocsr()
    deg      = np.asarray(A.sum(axis=1)).ravel()
    d_inv_sq = np.where(deg > 0, 1.0 / np.sqrt(deg + 1e-8), 0.)
    An       = A.multiply(d_inv_sq[:, np.newaxis]).multiply(d_inv_sq[np.newaxis, :]).tocoo()
    idx  = torch.from_numpy(np.stack([An.row, An.col]).astype(np.int64))
    vals = torch.from_numpy(An.data.astype(np.float32))
    return torch.sparse_coo_tensor(idx, vals, size=(N, N)).coalesce()


def eval_lgcn(model, adj, val_u, top_items_list, X_eval):
    model.eval()
    with torch.no_grad():
        ue, ie = model.forward(adj)
    ue_np = ue.numpy(); ie_np = ie.numpy()
    nd_list = []
    for uid in val_u:
        if uid not in user2idx: continue
        ui  = user2idx[uid]
        ti  = test_items_by_user.get(uid, set())
        if not ti: continue
        x_u = np.asarray(X_eval.getrow(ui).todense(), dtype=np.float32).ravel()
        sc  = ue_np[ui] @ ie_np.T
        sc[x_u > 0] = -1.
        top = np.argpartition(sc, -10)[-10:]; top = top[np.argsort(sc[top])[::-1]]
        nd_list.append(ndcg_at_k([top_items_list[ix] for ix in top], ti, 10))
    return float(np.mean(nd_list)) if nd_list else 0.


print('LightGCN, build_normalized_adj, eval_lgcn definidos.')


In [ ]:
print('Construyendo adj normalizado...')
t0 = time.time()
adj_norm = build_normalized_adj(X_top_td, n_u, N_TOP)
print(f'  adj_norm: {adj_norm.shape}  nnz={adj_norm._nnz():,}  [{time.time()-t0:.1f}s]')

X_top_td_coo   = X_top_td.tocoo()
pos_users_lgcn = X_top_td_coo.row.astype(np.int64)
pos_items_lgcn = X_top_td_coo.col.astype(np.int64)
print(f'Pares positivos: {len(pos_users_lgcn):,}')

cfg = {'emb_dim': 32, 'n_layers': 1, 'lr': 1e-3, 'bs': 1024,
       'epochs': 50, 'eval_every': 10, 'patience': 5, 'l2_reg': 1e-4}

torch.manual_seed(RANDOM_STATE)
model_lgcn = LightGCN(n_u, N_TOP, cfg['emb_dim'], cfg['n_layers'])
optimizer  = torch.optim.Adam(model_lgcn.parameters(), lr=cfg['lr'])
rng_lgcn   = np.random.default_rng(RANDOM_STATE)

best_lgcn_val   = 0.; best_lgcn_ep = 0; patience_cnt = 0
converge_flag   = True; lgcn_history = []
best_lgcn_state = None

print(f'{"Epoch":>6}  {"BPR loss":>10}  {"NDCG@10 val":>12}  {"t(s)":>7}')
print('-' * 45)

for epoch in range(1, cfg['epochs'] + 1):
    t_ep = time.time()
    model_lgcn.train()
    perm      = rng_lgcn.permutation(len(pos_users_lgcn))
    total_loss = 0.; n_batches = 0
    for start in range(0, len(pos_users_lgcn), cfg['bs']):
        idx    = perm[start:start+cfg['bs']]
        u_b    = torch.from_numpy(pos_users_lgcn[idx])
        p_b    = torch.from_numpy(pos_items_lgcn[idx])
        neg_b  = torch.randint(0, N_TOP, (len(idx),))
        loss   = model_lgcn.bpr_loss(adj_norm, u_b, p_b, neg_b, cfg['l2_reg'])
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item(); n_batches += 1
    avg_loss = total_loss / n_batches if n_batches > 0 else 0.
    ep_time  = time.time() - t_ep

    if epoch % cfg['eval_every'] == 0 or epoch == 1:
        ndcg_v = eval_lgcn(model_lgcn, adj_norm, val_users[:min(100, len(val_users))],
                            top_items_global, X_top_td)
        lgcn_history.append({'epoch': epoch, 'bpr_loss': avg_loss, 'ndcg10_val': ndcg_v})
        print(f'  {epoch:6d}  {avg_loss:10.4f}  {ndcg_v:12.5f}  {ep_time:6.1f}s')
        if ndcg_v > best_lgcn_val:
            best_lgcn_val = ndcg_v; best_lgcn_ep = epoch; patience_cnt = 0
            best_lgcn_state = {k: v.clone() for k, v in model_lgcn.state_dict().items()}
        else:
            patience_cnt += 1
            if patience_cnt >= cfg['patience']:
                print(f'  Early stopping en epoch {epoch}')
                break
        if epoch >= 30 and ndcg_v < 0.005:
            print(f'  WARNING: NDCG@10_val={ndcg_v:.5f} < 0.005 en epoch {epoch}')
            print('  LightGCN NO CONVERGE ? pasando a Estrategia 4')
            converge_flag = False
            break

print(f'Mejor NDCG@10_val: {best_lgcn_val:.5f} en epoch {best_lgcn_ep}')
print(f'Convergencia: {"Si" if converge_flag else "No"}')


In [ ]:
if converge_flag and best_lgcn_val > 0.010:
    if best_lgcn_state: model_lgcn.load_state_dict(best_lgcn_state)
    model_lgcn.eval()
    with torch.no_grad():
        uf_emb, if_emb = model_lgcn.forward(adj_norm)
    uf_np = uf_emb.numpy(); if_np = if_emb.numpy()

    def get_lgcn(uid, n):
        if uid not in user2idx: return []
        ui  = user2idx[uid]
        x_u = np.asarray(X_top_td.getrow(ui).todense(), dtype=np.float32).ravel()
        sc  = uf_np[ui] @ if_np.T; sc[x_u > 0] = -1.
        top = np.argpartition(sc, -n)[-n:]; top = top[np.argsort(sc[top])[::-1]]
        return [top_items_global[ix] for ix in top]

    print(f'Evaluando LightGCN+TD en test ({len(test_users_b)})...')
    t0 = time.time()
    m_lgcn_test = evaluate(
        get_lgcn, test_users_b, test_items_by_user, test_tx_by_user,
        item_pop_dict, n_total_train, n_items_global, baseline_conv
    )
    print(f'  [{time.time()-t0:.1f}s]')
    ndcg_lgcn_test = m_lgcn_test['NDCG@10']
    d3 = (ndcg_lgcn_test - NDCG_NB12B) / NDCG_NB12B * 100
    print(f'RP3+TD       : NDCG@10={NDCG_NB12B:.5f}')
    print(f'LightGCN+TD  : NDCG@10={ndcg_lgcn_test:.5f}  ({"+" if d3>=0 else ""}{d3:.1f}%)')
    torch.save({'model_state_dict': model_lgcn.state_dict(),
                'config': cfg, 'ndcg10_test': ndcg_lgcn_test,
                'ndcg10_val': best_lgcn_val},
               ENC_DIR / 'lightgcn_td_best.pt')
    print(f'Guardado: {ENC_DIR / "lightgcn_td_best.pt"}')
    lgcn_available = True
    uf_np_lgcn = uf_np; if_np_lgcn = if_np

else:
    print('LGCN NO CONVERGIO')
    print(f'  NDCG@10_val max: {best_lgcn_val:.5f}')
    print('  Causa: 57.5% de usuarios con solo 1 item en train')
    print('  El grafo es demasiado sparse para propagacion de embeddings util.')
    print('  LightGCN necesita min ~5 interacciones/usuario (Wang et al. 2019).')
    ndcg_lgcn_test = float('nan')
    lgcn_available = False
    m_lgcn_test    = {}
    uf_np_lgcn     = None
    if_np_lgcn     = None


## Estrategia 4 ? Ensemble Spearman

Seleccionar los 3 modelos con menor correlacion mutua (Spearman sobre rankings).
Optimizar pesos con Optuna (40 trials).

**Regla fija:** EASE^R SIEMPRE con ?=500 ? ?=14.3 no generalizo en NB12b.


In [ ]:
print('Construyendo EASE^R(lambda=500)...')
t0 = time.time()
G_sp      = X_top_csr.T @ X_top_csr
G_dense   = np.asarray(G_sp.todense(), dtype=np.float32)
del G_sp; gc.collect()
diag_idx  = np.arange(N_TOP)
G_reg500  = G_dense.copy()
G_reg500[diag_idx, diag_idx] += EASE_LAM_500
B_inv500  = np.linalg.inv(G_reg500)
del G_reg500
d_inv500  = np.diag(B_inv500).copy()
B_ease500 = -(B_inv500 / d_inv500[np.newaxis, :]).astype(np.float32)
np.fill_diagonal(B_ease500, 0.)
del B_inv500, G_dense; gc.collect()
print(f'  B_ease500: {B_ease500.shape}  [{time.time()-t0:.1f}s]')

def make_get_ease(B):
    def get_fn(uid, n):
        if uid not in user2idx: return []
        ui  = user2idx[uid]
        row = X_top_csr.getrow(ui)
        x_u = np.asarray(row.todense(), dtype=np.float32).ravel()
        sc  = x_u @ B; sc[x_u > 0] = -1.
        top = np.argpartition(sc, -n)[-n:]; top = top[np.argsort(sc[top])[::-1]]
        return [top_items_global[ix] for ix in top]
    return get_fn

get_ease500 = make_get_ease(B_ease500)

candidatos = {
    'rp3_td':     (get_rp3_td,   W_rp3_td,    X_top_td,   'RP3beta+TD'),
    'ease_500':   (get_ease500,  B_ease500,   X_top_csr,  'EASE^R(lam=500)'),
    'rp3_td_ips': (get_ips_best, W_rp3_td_ips, X_top_td_ips, 'RP3+TD+IPS'),
    'rp3_mb_td':  (get_mb_best,  W_rp3_mb,    X_mb_best,  'RP3+MB+TD'),
}
if lgcn_available:
    def get_lgcn_c(uid, n):
        if uid not in user2idx: return []
        ui  = user2idx[uid]
        x_u = np.asarray(X_top_td.getrow(ui).todense(), dtype=np.float32).ravel()
        sc  = uf_np_lgcn[ui] @ if_np_lgcn.T; sc[x_u > 0] = -1.
        top = np.argpartition(sc, -n)[-n:]; top = top[np.argsort(sc[top])[::-1]]
        return [top_items_global[ix] for ix in top]
    candidatos['lightgcn_td'] = (get_lgcn_c, None, X_top_td, 'LightGCN+TD')

print(f'Candidatos al ensemble: {list(candidatos.keys())}')


In [ ]:
print('Calculando scores de candidatos en val_users...')
scores_all = {}

for name, (get_fn, W_m, X_m, label) in candidatos.items():
    t0 = time.time()
    model_scores = []
    for uid in val_users:
        if uid not in user2idx:
            model_scores.append(np.zeros(N_TOP, dtype=np.float32)); continue
        ui  = user2idx[uid]
        x_u = np.asarray(X_m.getrow(ui).todense(), dtype=np.float32).ravel() if hasattr(X_m, 'getrow') else np.zeros(N_TOP)
        if name == 'ease_500':
            sc = x_u @ B_ease500
        elif name == 'lightgcn_td' and lgcn_available:
            sc = uf_np_lgcn[ui] @ if_np_lgcn.T
        elif W_m is not None and isinstance(W_m, np.ndarray):
            sc = x_u @ W_m
        else:
            sc = np.zeros(N_TOP, dtype=np.float32)
        sc[x_u > 0] = -np.inf
        model_scores.append(sc.astype(np.float32))
    scores_all[name] = np.vstack(model_scores)
    print(f'  {name:15s}: {scores_all[name].shape}  [{time.time()-t0:.1f}s]')

model_names = list(scores_all.keys())
n_models    = len(model_names)
corr_matrix = np.zeros((n_models, n_models))

for ii, m1 in enumerate(model_names):
    for jj, m2 in enumerate(model_names):
        if ii == jj: corr_matrix[ii, jj] = 1.0; continue
        if ii > jj:  corr_matrix[ii, jj] = corr_matrix[jj, ii]; continue
        rhos = []
        for u_idx in range(len(val_users)):
            s1 = scores_all[m1][u_idx]; s2 = scores_all[m2][u_idx]
            mask = np.isfinite(s1) & np.isfinite(s2)
            if mask.sum() < 10: continue
            rho, _ = spearmanr(s1[mask], s2[mask])
            if not np.isnan(rho): rhos.append(rho)
        rho_avg = float(np.mean(rhos)) if rhos else 0.
        corr_matrix[ii, jj] = corr_matrix[jj, ii] = rho_avg

df_corr = pd.DataFrame(corr_matrix, index=model_names, columns=model_names)
print('\nMatriz de correlaciones Spearman:')
print(df_corr.round(3).to_string())

if n_models >= 3:
    min_corr = float('inf'); best_trio = None
    for trio in combinations(range(n_models), 3):
        avg_c = np.mean([corr_matrix[trio[a], trio[b]] for a, b in combinations(range(3), 2)])
        if avg_c < min_corr: min_corr = avg_c; best_trio = trio
    selected = [model_names[ix] for ix in best_trio]
    print(f'\nTrio seleccionado (corr_promedio={min_corr:.3f}): {selected}')
elif n_models >= 2:
    selected = model_names[:2]
    print(f'Solo 2 modelos: {selected}')
else:
    selected = model_names
    print(f'Solo 1 modelo: {selected}')


In [ ]:
val_user_to_idx = {uid: idx for idx, uid in enumerate(val_users)}

sel_scores_val = {}
for name in selected:
    scr = scores_all[name].copy()
    for u_idx in range(scr.shape[0]):
        mask = np.isfinite(scr[u_idx])
        if mask.any(): scr[u_idx][mask] = minmax_norm(scr[u_idx][mask])
        scr[u_idx][~mask] = 0.
    sel_scores_val[name] = scr

def make_ensemble_fn(w_dict, scores_dict, u2idx, top_items_list):
    def get_fn(uid, n):
        if uid not in u2idx: return []
        ui = u2idx[uid]
        sc = sum(w * scores_dict[nm][ui] for nm, w in w_dict.items() if nm in scores_dict)
        top = np.argpartition(sc, -n)[-n:]; top = top[np.argsort(sc[top])[::-1]]
        return [top_items_list[ix] for ix in top]
    return get_fn

N_TRIALS_ENS = 40

def objective_ens(trial):
    if len(selected) >= 3:
        w1 = trial.suggest_float('w1', 0.2, 0.75)
        w2 = trial.suggest_float('w2', 0.05, 0.5)
        w3 = 1.0 - w1 - w2
        if not (0.05 <= w3 <= 0.6): return 0.0
        w_dict = {selected[0]: w1, selected[1]: w2, selected[2]: w3}
    elif len(selected) == 2:
        w1 = trial.suggest_float('w1', 0.1, 0.9)
        w_dict = {selected[0]: w1, selected[1]: 1.0-w1}
    else:
        w_dict = {selected[0]: 1.0}
    get_fn = make_ensemble_fn(w_dict, sel_scores_val, val_user_to_idx, top_items_global)
    m = evaluate(get_fn, val_users, test_items_by_user, test_tx_by_user,
                 item_pop_dict, n_total_train, n_items_global, baseline_conv)
    return m.get('NDCG@10', 0.)

print(f'Optuna Ensemble ({N_TRIALS_ENS} trials)...')
t_opt = time.time()
study_ens = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_ens.optimize(objective_ens, n_trials=N_TRIALS_ENS, show_progress_bar=True)
print(f'Optuna Ensemble finalizado en {time.time()-t_opt:.0f}s')

best_ens_val    = study_ens.best_value
best_ens_params = study_ens.best_params
print(f'NDCG@10_val={best_ens_val:.5f}  params={best_ens_params}')


In [ ]:
print('Calculando scores de test para ensemble final...')
test_user_to_idx = {uid: idx for idx, uid in enumerate(test_users_b)}
sel_scores_test  = {}

for name in selected:
    t0  = time.time()
    _, W_m, X_m, _ = candidatos[name]
    scr_test = []
    for uid in test_users_b:
        if uid not in user2idx:
            scr_test.append(np.zeros(N_TOP, dtype=np.float32)); continue
        ui  = user2idx[uid]
        x_u = np.asarray(X_m.getrow(ui).todense(), dtype=np.float32).ravel() if hasattr(X_m, 'getrow') else np.zeros(N_TOP)
        if name == 'ease_500':
            sc = x_u @ B_ease500
        elif name == 'lightgcn_td' and lgcn_available:
            sc = uf_np_lgcn[ui] @ if_np_lgcn.T
        elif W_m is not None and isinstance(W_m, np.ndarray):
            sc = x_u @ W_m
        else:
            sc = np.zeros(N_TOP, dtype=np.float32)
        sc[x_u > 0] = -np.inf
        mask = np.isfinite(sc)
        sc_n = np.zeros(N_TOP, dtype=np.float32)
        if mask.any(): sc_n[mask] = minmax_norm(sc[mask])
        scr_test.append(sc_n)
    sel_scores_test[name] = np.vstack(scr_test)
    print(f'  {name:15s}: {sel_scores_test[name].shape}  [{time.time()-t0:.1f}s]')

if len(selected) >= 3:
    w1_f = best_ens_params.get('w1', 0.5); w2_f = best_ens_params.get('w2', 0.3)
    w3_f = 1.0 - w1_f - w2_f
    w_final = {selected[0]: w1_f, selected[1]: w2_f, selected[2]: w3_f}
elif len(selected) == 2:
    w1_f = best_ens_params.get('w1', 0.5)
    w_final = {selected[0]: w1_f, selected[1]: 1.0-w1_f}
else:
    w_final = {selected[0]: 1.0}

get_ens_test = make_ensemble_fn(w_final, sel_scores_test, test_user_to_idx, top_items_global)

print(f'Evaluando Ensemble Final en test ({len(test_users_b)})...')
t0 = time.time()
m_ens_test = evaluate(
    get_ens_test, test_users_b, test_items_by_user, test_tx_by_user,
    item_pop_dict, n_total_train, n_items_global, baseline_conv
)
print(f'  [{time.time()-t0:.1f}s]')
ndcg_ens_test = m_ens_test['NDCG@10']
d4 = (ndcg_ens_test - NDCG_NB12B) / NDCG_NB12B * 100
print(f'RP3+TD (base)     : NDCG@10={NDCG_NB12B:.5f}')
print(f'Ensemble Spearman : NDCG@10={ndcg_ens_test:.5f}  ({"+" if d4>=0 else ""}{d4:.1f}%)')
print(f'Target 0.030      : {"SUPERADO" if ndcg_ens_test >= TARGET_NDCG else "no alcanzado"}')


## ?Hay un techo estructural en RetailRocket?

El 57.5% de los usuarios evaluados tiene exactamente 1 item en train.
Ningun modelo puede predecir el siguiente item de un usuario que solo ha visto 1 producto.

| Familia | Mejor NDCG@10 |
|---------|---------------|
| CF clasico (SVD, ItemKNN) | 0.0081 |
| EASE^R (optimo) | 0.0260 |
| RP3beta (optimo) | 0.0258 |
| RP3beta + Temporal Decay | **0.02859** |
| NB14 estrategias | (ver tabla) |

**Techo practico (protocolo ?1 interaccion):** ~0.028?0.033

Para superar ese rango se requiere:
- Modelos de sesion (GRU4Rec con contexto de clic activo)
- Filtrado estricto ?5 interacciones (cambia el protocolo)
- Contexto externo (precio, imagenes, busqueda del usuario)


In [ ]:
import os

def fmt_delta(val, base):
    if base == 0 or (isinstance(val, float) and np.isnan(val)): return '?'
    d = (val - base) / base * 100
    return f'{"+" if d>=0 else ""}{d:.1f}%'

def delta_030(val):
    if isinstance(val, float) and np.isnan(val): return '?'
    d = (val - TARGET_NDCG) / TARGET_NDCG * 100
    return f'{"+" if d>=0 else ""}{d:.1f}%'

rows_res = [
    {'Modelo': 'RP3beta+TD (NB12b baseline)',
     'NDCG@10': NDCG_NB12B, 'Delta_vs_base': 'baseline', 'Delta_0030': delta_030(NDCG_NB12B)},
    {'Modelo': f'RP3+TD+IPS(s={best_smoothing})',
     'NDCG@10': ndcg_ips_test, 'Delta_vs_base': fmt_delta(ndcg_ips_test, NDCG_NB12B), 'Delta_0030': delta_030(ndcg_ips_test)},
    {'Modelo': f'RP3+MB+TD',
     'NDCG@10': ndcg_mb_test, 'Delta_vs_base': fmt_delta(ndcg_mb_test, NDCG_NB12B), 'Delta_0030': delta_030(ndcg_mb_test)},
    {'Modelo': f'LightGCN+TD ({"convergio" if lgcn_available else "NO convergio"})',
     'NDCG@10': ndcg_lgcn_test if not np.isnan(ndcg_lgcn_test) else None,
     'Delta_vs_base': fmt_delta(ndcg_lgcn_test, NDCG_NB12B) if not np.isnan(ndcg_lgcn_test) else '?',
     'Delta_0030': delta_030(ndcg_lgcn_test) if not np.isnan(ndcg_lgcn_test) else '?'},
    {'Modelo': f'Ensemble Spearman',
     'NDCG@10': ndcg_ens_test, 'Delta_vs_base': fmt_delta(ndcg_ens_test, NDCG_NB12B), 'Delta_0030': delta_030(ndcg_ens_test)},
]

df_res = pd.DataFrame(rows_res)
print('=' * 80)
print('TABLA COMPARATIVA NB14')
print('=' * 80)
print(df_res.to_string(index=False))

out_csv14 = DATA_DIR / 'model_comparison_nb14.csv'
df_res.to_csv(out_csv14, index=False)
print(f'Guardado: {out_csv14}')

all_ndcgs = [v for v in [ndcg_ips_test, ndcg_mb_test, ndcg_ens_test,
                          ndcg_lgcn_test if not np.isnan(ndcg_lgcn_test) else 0.] if v > 0]
best_nb14  = max(all_ndcgs) if all_ndcgs else NDCG_NB12B
best_proj  = max(best_nb14, NDCG_NB12B)

print(f'\nMEJOR NB14  : NDCG@10={best_nb14:.5f}')
print(f'MEJOR PROYECTO: NDCG@10={best_proj:.5f}')
print(f'{"TARGET 0.030 SUPERADO" if best_nb14 >= TARGET_NDCG else "Target 0.030 no alcanzado"}')


In [ ]:
# Actualizar model_comparison_final.csv
final_csv = DATA_DIR / 'model_comparison_final.csv'
df_final  = pd.read_csv(final_csv)
cur_best  = df_final['NDCG@10'].max() if 'NDCG@10' in df_final.columns else 0.

if best_nb14 > cur_best:
    new_row = pd.DataFrame([{'NDCG@10': best_nb14, 'Notebook': 'NB14',
                              'Modelo': f'NB14-best'}])
    df_final = pd.concat([df_final, new_row], ignore_index=True)
    df_final.to_csv(final_csv, index=False)
    print(f'model_comparison_final.csv actualizado (NDCG@10={best_nb14:.5f})')
else:
    print(f'Campeon actual no superado ({cur_best:.5f}) ? sin cambios')

# Actualizar model_justification.md
doc_path = DOCS_DIR / 'model_justification.md'
doc_text  = doc_path.read_text(encoding='utf-8')
MARKER57  = '## 5.7'

lcn_line = (f'NDCG@10 test={ndcg_lgcn_test:.5f}' if lgcn_available
            else 'NO convergio ? grafo demasiado sparse (57.5% usuarios con 1 item en train)')

sec57 = (
    f'\n\n## 5.7 NB14 ? Estrategias Avanzadas\n\n'
    f'### 5.7.1 IPS + TD\n'
    f'smoothing optimo={best_smoothing}  NDCG@10 test={ndcg_ips_test:.5f}  '
    f'Delta vs NB12b={fmt_delta(ndcg_ips_test, NDCG_NB12B)}\n\n'
    f'### 5.7.2 Multi-Behavior + TD\n'
    f'Pesos optimos: w_view={best_mb["w_view"]:.3f} w_cart={best_mb["w_cart"]:.3f} w_trans={best_mb["w_trans"]:.3f}\n'
    f'NDCG@10 test={ndcg_mb_test:.5f}  Delta vs NB12b={fmt_delta(ndcg_mb_test, NDCG_NB12B)}\n\n'
    f'### 5.7.3 LightGCN + TD\n'
    f'{lcn_line}\n\n'
    f'### 5.7.4 Ensemble Spearman\n'
    f'Modelos: {selected}\n'
    f'NDCG@10 test={ndcg_ens_test:.5f}  Delta vs NB12b={fmt_delta(ndcg_ens_test, NDCG_NB12B)}\n\n'
    f'### 5.7.5 Techo estructural\n'
    f'57.5% usuarios con 1 item en train. Techo practico: ~0.028-0.033.\n'
    f'Mejor resultado NB14: NDCG@10={best_nb14:.5f}  '
    f'Mejor proyecto: NDCG@10={best_proj:.5f}\n'
)

if MARKER57 not in doc_text:
    doc_path.write_text(doc_text + sec57, encoding='utf-8')
    print('model_justification.md ? seccion 5.7 aniadida')
else:
    idx0 = doc_text.index(MARKER57)
    nxt  = doc_text.find('\n## ', idx0 + 10)
    doc_text = (doc_text[:idx0] + sec57.strip() + '\n\n' + doc_text[nxt:]
                if nxt > 0 else doc_text[:idx0] + sec57.strip())
    doc_path.write_text(doc_text, encoding='utf-8')
    print('model_justification.md ? seccion 5.7 reemplazada')

# Actualizar EXPLICACION_ORAL.md
oral_path = DOCS_DIR / 'EXPLICACION_ORAL.md'
oral_text  = oral_path.read_text(encoding='utf-8')
BLOQUE7    = '## BLOQUE 7'

lcn_oral = ('LightGCN convergio. NDCG@10=' + f'{ndcg_lgcn_test:.5f}'
            if lgcn_available
            else 'LightGCN NO convergio ? 57.5% usuarios con solo 1 item en train.')

bloque7 = (
    f'\n\n---\n\n## BLOQUE 7 ? Estrategias Avanzadas (NB14)\n\n'
    f'### Estrategias implementadas\n'
    f'1. IPS+TD ? debiasing popularidad\n'
    f'2. Multi-Behavior+TD ? pesos view/cart/trans con Optuna\n'
    f'3. LightGCN+TD ? embeddings GCN sobre grafo con TD\n'
    f'4. Ensemble Spearman ? componentes por correlacion\n\n'
    f'### IPS\n'
    f'Divide la senal de cada item por su popularidad. smoothing optimo={best_smoothing}.\n'
    f'NDCG@10={ndcg_ips_test:.5f} Delta={fmt_delta(ndcg_ips_test, NDCG_NB12B)}\n\n'
    f'### Multi-Behavior\n'
    f'Optuna aprende w_view={best_mb["w_view"]:.3f} w_cart={best_mb["w_cart"]:.3f} w_trans={best_mb["w_trans"]:.3f}.\n'
    f'NDCG@10={ndcg_mb_test:.5f} Delta={fmt_delta(ndcg_mb_test, NDCG_NB12B)}\n\n'
    f'### LightGCN\n''{lcn_oral}\n\n'
    f'### Ensemble Spearman\n'
    f'Modelos seleccionados: {selected}\n'
    f'NDCG@10={ndcg_ens_test:.5f} Delta={fmt_delta(ndcg_ens_test, NDCG_NB12B)}\n\n'
    f'### Conclusion\n'
    f'Mejor NB14: NDCG@10={best_nb14:.5f}. '
    f'Techo del dataset con protocolo est: ~0.028-0.033.\n'
)

if BLOQUE7 not in oral_text:
    oral_path.write_text(oral_text + bloque7, encoding='utf-8')
    print('EXPLICACION_ORAL.md ? Bloque NB14 aniadido')
else:
    print('EXPLICACION_ORAL.md ? Bloque NB14 ya existe')

# requirements.txt
req_path = ROOT / 'requirements.txt'
req_text  = req_path.read_text(encoding='utf-8')
if 'torch' not in req_text:
    req_path.write_text(req_text + '\ntorch>=2.0.0\n', encoding='utf-8')
    print('requirements.txt ? torch aniadido')
else:
    print('requirements.txt ? torch ya presente')


In [ ]:
print()
print('=' * 70)
print('NB14 ? REPORTE FINAL')
print('=' * 70)
print(f'Punto de partida (RP3+TD NB12b): NDCG@10={NDCG_NB12B:.5f}')
print(f'Target                          : NDCG@10={TARGET_NDCG:.5f}')
print()
print(f'E1 RP3+TD+IPS(s={best_smoothing})       : {ndcg_ips_test:.5f}  {fmt_delta(ndcg_ips_test, NDCG_NB12B)}')
print(f'E2 RP3+MB+TD                    : {ndcg_mb_test:.5f}  {fmt_delta(ndcg_mb_test, NDCG_NB12B)}')
lgcn_str = f'{ndcg_lgcn_test:.5f}' if not np.isnan(ndcg_lgcn_test) else 'NO CONVERGIO'
print(f'E3 LightGCN+TD                  : {lgcn_str}')
print(f'E4 Ensemble Spearman             : {ndcg_ens_test:.5f}  {fmt_delta(ndcg_ens_test, NDCG_NB12B)}')
print()
print(f'MEJOR RESULTADO NB14   : {best_nb14:.5f}')
print(f'MEJOR RESULTADO PROYECTO: {best_proj:.5f}')
print()
if best_nb14 >= TARGET_NDCG:
    print(f'TARGET 0.030 SUPERADO  gap={best_nb14 - TARGET_NDCG:.5f} pts')
else:
    print(f'Target 0.030 no alcanzado.  Gap: {TARGET_NDCG - best_nb14:.5f} pts')
    print('  Techo estructural del dataset (~0.028-0.033) probablemente alcanzado.')
print('=' * 70)
